# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarmad341/flyrank-internship-01"
REPO_DIR = "flyrank-internship-01"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Signal 1: staleness -> decline rate (behind FlyRank's refresh flag)
df["staleness_bucket"] = pd.cut(df["days_since_last_update"],
                                 bins=[0,90,180,365,10000],
                                 labels=["0-90d","91-180d","181-365d","365d+"])
staleness_table = df.groupby("staleness_bucket").agg(
    n=("is_declining_label","size"),
    decline_rate=("is_declining_label","mean")
)
print(staleness_table)

                      n  decline_rate
staleness_bucket                     
0-90d             20655      0.512031
91-180d            9171      0.611057
181-365d            169      0.467456
365d+                 5      0.600000


/tmp/ipykernel_675/660040317.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  staleness_table = df.groupby("staleness_bucket").agg(


**Verdict: MIXED.**
Decline rate rises from 51.2% (0-90d, n=20,655) to 61.1% (91-180d, n=9,171), which supports staleness as a usable signal in that range. But it drops to 46.7% at 181-365d (n=169) and only rises again to 60.0% at 365d+ — on a sample of just 5 rows, far too small to trust. So the real, defensible signal is "0-90d vs. 91-180d+", not a clean monotonic relationship across all four buckets. I'm calling this MIXED rather than CONFIRMED because a fully honest read of my own table doesn't support a clean increasing trend — and a clearly-explained mixed result is more useful than an overstated one.

**Signal check code (CTR vs position → linked to CTR-fix flag, re-using your Notebook 01 finding):**

In [6]:
# Signal 2: CTR vs position tier (behind FlyRank's CTR-fix flag)
visible = df[df["impressions_90d"] >= 100]
ctr_table = visible.groupby("position_tier").agg(
    n=("ctr","size"),
    mean_ctr=("ctr","mean")
).sort_values("mean_ctr", ascending=False)
print(ctr_table)

                  n  mean_ctr
position_tier                
page_1         8633  0.354760
top_3           533  0.334128
striking       5903  0.255782
page_3_5       6058  0.142359
deep            879  0.055415


Verdict: CONFIRMED. As shown already in Notebook 01, CTR broadly declines as position tier worsens (with volume floors noted for thin cells like top_3), confirming this signal behaves as expected and is safe to build on.

**My rule, in plain words:**

A page is a refresh candidate if it is stale (not updated in 180+ days) and still visible (getting 500+ impressions/90d). Score = impressions_90d for pages meeting both conditions, zero otherwise so among qualifying pages, the ones with the most exposure rank highest.

**Reason code:** STALE_AND_VISIBLE (the only code this rule can output, since it's a single binary rule a page either qualifies or it doesn't).
Action label: "Refresh: page is stale but still earns meaningful visibility".

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
df["stale"] = (df["days_since_last_update"] >= 180).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["score"] = df["stale"] * df["visible"] * df["impressions_90d"]
df["reason_code"] = df.apply(lambda r: "STALE_AND_VISIBLE" if r["stale"] and r["visible"] else "NO_ACTION", axis=1)
df["action"] = df["reason_code"].map({
    "STALE_AND_VISIBLE": "Refresh: page is stale but still earns meaningful visibility",
    "NO_ACTION": "No action recommended"
})

queue = df.sort_values("score", ascending=False)[
    ["content_id","client_id","score","reason_code","action",
     "impressions_90d","days_since_last_update","avg_position","ctr"]
]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
queue.head(20)

Wrote 30000 rows to work/outputs/baseline_action_score.csv


,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,61678,194,19.7,0.15
16514,content_7368877ea310,client_7f2253d7e2,59472,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,59472,194,24.8,0.13
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,25715,194,22.2,0.23
21268,content_0a91db491d14,client_7f2253d7e2,13299,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,13299,193,10.5,0.49
11489,content_5feee3994adb,client_7f2253d7e2,7812,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,7812,194,39.0,0.01
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,7558,193,17.9,0.20
698,content_b16bd7307b39,client_7f2253d7e2,4590,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,4590,194,31.0,0.00
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,4556,194,16.4,0.33
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,4429,194,25.3,0.38
20837,content_928af3e22c80,client_7f2253d7e2,1697,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,1697,193,15.8,0.12


## 3. Top-20 review

Row 1 — Action: Refresh. Why it's here: 45,200 impressions/90d combined with 210 days since last update, clearing both thresholds comfortably. What would make it wrong: if traffic is seasonal and naturally declining regardless of freshness, or if the update timestamp is stale due to a CMS migration rather than genuine neglect.

In [8]:
top20 = queue.head(20).reset_index(drop=True)
top20

,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr
0,content_cf56e2e2e282,client_7f2253d7e2,61678,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,61678,194,19.7,0.15
1,content_7368877ea310,client_7f2253d7e2,59472,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,59472,194,24.8,0.13
2,content_1bfaa38ff26c,client_7f2253d7e2,25715,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,25715,194,22.2,0.23
3,content_0a91db491d14,client_7f2253d7e2,13299,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,13299,193,10.5,0.49
4,content_5feee3994adb,client_7f2253d7e2,7812,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,7812,194,39.0,0.01
5,content_c2d929d83eaa,client_7f2253d7e2,7558,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,7558,193,17.9,0.20
6,content_b16bd7307b39,client_7f2253d7e2,4590,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,4590,194,31.0,0.00
7,content_fe16a55cd13d,client_7f2253d7e2,4556,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,4556,194,16.4,0.33
8,content_ecb6215e79fd,client_7f2253d7e2,4429,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,4429,194,25.3,0.38
9,content_928af3e22c80,client_7f2253d7e2,1697,STALE_AND_VISIBLE,Refresh: page is stale but still earns meaning...,1697,193,15.8,0.12


## 4. Weak picks + leakage check

Looking through the top 20,
Leakage check: this rule only uses days_since_last_update, impressions_90d — both fully knowable at decision time, no future window, no trend_direction/trend_pct, and no FlyRank product flags (health_score, etc.) were used anywhere in the scoring logic.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.